# Intro: why compile?
By default, torch executes operations eagerly.  It computes `sin` over all of `x`, then `cos` over all of `y`, and so on

In [1]:
import torch

In [2]:
def f(x):
    y = x.sin()
    z = y.cos()
    return z

f(torch.randn(3, 3))

tensor([[0.9968, 0.5963, 0.7659],
        [0.6506, 0.7535, 0.9947],
        [0.9685, 0.7639, 0.7363]])

What happens if we instead compile this function?

In [3]:
torch._logging.set_logs(graph_code=True)

comp_f = torch.compile(f)
comp_f(torch.randn(3, 3))

V1006 16:00:11.236000 14541 torch/_dynamo/output_graph.py:2936] [0/0] [__graph_code] TRACED GRAPH
V1006 16:00:11.236000 14541 torch/_dynamo/output_graph.py:2936] [0/0] [__graph_code]  ===== __compiled_fn_1_0fd580b2_50e3_41dc_a294_3c5994d61f6d =====
V1006 16:00:11.236000 14541 torch/_dynamo/output_graph.py:2936] [0/0] [__graph_code]  /home/rob/code/scratch/pytorch/presentation/env/lib/python3.14/site-packages/torch/fx/_lazy_graph_module.py class GraphModule(torch.nn.Module):
V1006 16:00:11.236000 14541 torch/_dynamo/output_graph.py:2936] [0/0] [__graph_code]     def forward(self, L_x_: "f32[3, 3][3, 1]cpu"):
V1006 16:00:11.236000 14541 torch/_dynamo/output_graph.py:2936] [0/0] [__graph_code]         l_x_ = L_x_
V1006 16:00:11.236000 14541 torch/_dynamo/output_graph.py:2936] [0/0] [__graph_code] 
V1006 16:00:11.236000 14541 torch/_dynamo/output_graph.py:2936] [0/0] [__graph_code]         # File: /tmp/ipykernel_14541/1201477647.py:2 in f, code: y = x.sin()
V1006 16:00:11.236000 14541 torc

tensor([[0.9394, 0.8903, 0.5421],
        [0.5542, 0.8207, 0.8569],
        [0.8937, 0.9972, 0.7798]])

First we figure out what the function is doing, storing the results in a format called an "FX graph".  Then we can compile the grpah to executable code.  I'm running on CPU, so this gives the following C++ code (you can see this for yourself by running with `TORCH_COMPILE_DEBUG=1` and looking in the `output_code.py` file) 

```c++
#include <torch/csrc/inductor/cpp_prefix.h>
extern "C"  void  kernel(const float* in_ptr0,
                       float* out_ptr0)
{
    std::atomic<int> inductor_cpu_integer_div_error{0};
    inductor_cpu_integer_div_error_flag = &inductor_cpu_integer_div_error;
    {
        for(int64_t x0=static_cast<int64_t>(0L); x0<static_cast<int64_t>(9L); x0+=static_cast<int64_t>(8L))
        {
            {
                if(C10_LIKELY(x0 >= static_cast<int64_t>(0) && x0 < static_cast<int64_t>(8L)))
                {
                    auto tmp0 = at::vec::Vectorized<float>::loadu(in_ptr0 + static_cast<int64_t>(x0), static_cast<int64_t>(8));
                    auto tmp1 = tmp0.sin();
                    auto tmp2 = tmp1.cos();
                    tmp2.store(out_ptr0 + static_cast<int64_t>(x0));
                }
                if(C10_UNLIKELY(x0 >= static_cast<int64_t>(8L) && x0 < static_cast<int64_t>(9L)))
                {
                    auto tmp0 = at::vec::Vectorized<float>::loadu(in_ptr0 + static_cast<int64_t>(x0), static_cast<int64_t>(1L));
                    auto tmp1 = tmp0.sin();
                    auto tmp2 = tmp1.cos();
                    tmp2.store(out_ptr0 + static_cast<int64_t>(x0), static_cast<int64_t>(1L));
                }
            }
        }
    }
    inductor_cpu_integer_div_error_flag = nullptr;
    inductor_cpu_throw_if_integer_div_error(inductor_cpu_integer_div_error);
}
```

Note that it's vectorized, and that there are fewer memory accesses.  This matters even more on GPU!

# But how does it work?
If we only cared about compiling code that looks like this, torch.compile could be a lot simpler.  We could just track whenever a torch operation occurs and add it to the FX graph.  But what if we have code like this?

In [4]:
num_calls = 0

@torch.compile(backend='eager')
def f(x):
    global num_calls
    num_calls += 1
    y = x.sin()
    z = y.cos()
    return z

torch._logging.set_logs(graph_code=True, side_effects=True, bytecode=True)
f(torch.randn(3, 3))

V1006 16:00:11.561000 14541 torch/_dynamo/convert_frame.py:1726] [1/0] [__bytecode] ORIGINAL BYTECODE f /tmp/ipykernel_14541/167820838.py line 3 
V1006 16:00:11.561000 14541 torch/_dynamo/convert_frame.py:1726] [1/0] [__bytecode]   3           RESUME                   0
V1006 16:00:11.561000 14541 torch/_dynamo/convert_frame.py:1726] [1/0] [__bytecode] 
V1006 16:00:11.561000 14541 torch/_dynamo/convert_frame.py:1726] [1/0] [__bytecode]   6           LOAD_GLOBAL              0 (num_calls)
V1006 16:00:11.561000 14541 torch/_dynamo/convert_frame.py:1726] [1/0] [__bytecode]               LOAD_SMALL_INT           1
V1006 16:00:11.561000 14541 torch/_dynamo/convert_frame.py:1726] [1/0] [__bytecode]               BINARY_OP               13 (+=)
V1006 16:00:11.561000 14541 torch/_dynamo/convert_frame.py:1726] [1/0] [__bytecode]               STORE_GLOBAL             0 (num_calls)
V1006 16:00:11.561000 14541 torch/_dynamo/convert_frame.py:1726] [1/0] [__bytecode] 
V1006 16:00:11.561000 14541 to

tensor([[0.9209, 0.5467, 0.7368],
        [0.9957, 0.9572, 0.6373],
        [0.9874, 0.5415, 0.7529]])

Now our code has a side effect.  FX graphs are optimized for torch operations - they don't include non-torch side effects.  Note that the graph is the same, but we've captured the side effect separately.  We'd have no way of knowing this just by looking at which torch operators are called - we need to symbolically execute the function and see what it does.  This is the job of Dynamo.  It produces an FX graph which is sent to the other major component of torch.compile - Inductor, which generates executable code.

### Graph breaks
This isn't such a hard side effect, but python is a big language and dynamo can't understand all of it.  What happens then?

In [5]:
import pickle

@torch.compile(backend='eager')
def f(x):
    y = x.sin()
    pickle.dumps('something')
    z = y.cos()
    return z

torch._logging.set_logs(graph_code=True)
f(torch.randn(3, 3))

/home/rob/code/scratch/pytorch/presentation/env/lib/python3.14/site-packages/torch/_dynamo/variables/functions.py:2604: UserWarning: Dynamo does not know how to trace the builtin `_pickle.dumps.` This function is either a Python builtin (e.g. _warnings.warn) or a third-party C/C++ Python extension (perhaps created with pybind).
If it is a Python builtin, please file an issue on GitHub so the PyTorch team can add support for it and see the next case for a workaround.
If it is a third-party C/C++ Python extension, please either wrap it into a PyTorch-understood custom operator (see https://pytorch.org/tutorials/advanced/custom_ops_landing_page.html for more details) or, if it is traceable, use `torch.compiler.allow_in_graph`.
  torch._dynamo.utils.warn_once(explanation + "\n" + "\n".join(hints))
V1006 16:00:11.636000 14541 torch/_dynamo/output_graph.py:2936] [2/0_1] [__graph_code] TRACED GRAPH
V1006 16:00:11.636000 14541 torch/_dynamo/output_graph.py:2936] [2/0_1] [__graph_code]  ===== _

tensor([[0.7283, 0.9059, 0.7640],
        [0.9648, 0.9773, 0.9237],
        [0.9977, 0.9413, 0.8865]])

Notice that we now have two graphs instead of one.  Dynamo started to trace our function, then got to something it didn't understand (pickle).  This is a graph break - we stop capturing the current graph, then fall back to the normal python interpreter to execute the unknown portion, then resume tracing.  This is a very flexible approach, but it comes with a performance cost - we can no longer optimize between the two graphs.  We can use `fullgraph=True` to detect when this happens - this will error out instead of silently graph breaking.

In [6]:
@torch.compile(backend='eager', fullgraph=True)
def f(x):
    y = x.sin()
    pickle.dumps('something')
    z = y.cos()
    return z

f(torch.randn(3, 3))

Unsupported: Attempted to call function marked as skipped
  Explanation: Dynamo does not know how to trace the builtin `_pickle.dumps.` This function is either a Python builtin (e.g. _warnings.warn) or a third-party C/C++ Python extension (perhaps created with pybind).
  Hint: If it is a Python builtin, please file an issue on GitHub so the PyTorch team can add support for it and see the next case for a workaround.
  Hint: If it is a third-party C/C++ Python extension, please either wrap it into a PyTorch-understood custom operator (see https://pytorch.org/tutorials/advanced/custom_ops_landing_page.html for more details) or, if it is traceable, use `torch.compiler.allow_in_graph`.

  Developer debug context: module: _pickle, qualname: dumps, skip reason: cannot determine source file for _pickle (likely a C extension or builtin)

 For more details about this graph break, please visit: https://meta-pytorch.github.io/compile-graph-break-site/gb/gb0007.html

from user code:
   File "/tmp/i

There are many reasons that Dynamo might graph break.

- Python features that Dynamo doesn't support
- Native code
- Fundamental (I/O, data-dependent control flow, etc.)

Notably, the last category includes constructs that are not supported by FX graphs.  If your code includes these, you will need to either accept the graph breaks, or devote some effort to avoiding them.

### Recompiles
Recompiles are another problem you might run into.

In [7]:
@torch.compile(backend='eager')
def f(x, flag):
    if flag:
        return x.sin()
    return x.cos()

torch._logging.set_logs(graph_code=True)
f(torch.randn(3, 3), True)

V1006 16:00:24.255000 14541 torch/_dynamo/output_graph.py:2936] [5/0] [__graph_code] TRACED GRAPH
V1006 16:00:24.255000 14541 torch/_dynamo/output_graph.py:2936] [5/0] [__graph_code]  ===== __compiled_fn_12_11d15e86_f5cd_45eb_9291_ef6dee5e605d =====
V1006 16:00:24.255000 14541 torch/_dynamo/output_graph.py:2936] [5/0] [__graph_code]  /home/rob/code/scratch/pytorch/presentation/env/lib/python3.14/site-packages/torch/fx/_lazy_graph_module.py class GraphModule(torch.nn.Module):
V1006 16:00:24.255000 14541 torch/_dynamo/output_graph.py:2936] [5/0] [__graph_code]     def forward(self, L_x_: "f32[3, 3][3, 1]cpu"):
V1006 16:00:24.255000 14541 torch/_dynamo/output_graph.py:2936] [5/0] [__graph_code]         l_x_ = L_x_
V1006 16:00:24.255000 14541 torch/_dynamo/output_graph.py:2936] [5/0] [__graph_code] 
V1006 16:00:24.255000 14541 torch/_dynamo/output_graph.py:2936] [5/0] [__graph_code]         # File: /tmp/ipykernel_14541/2688769548.py:4 in f, code: return x.sin()
V1006 16:00:24.255000 14541 

tensor([[ 0.9699, -0.4205,  0.6746],
        [-0.3149,  0.1767, -0.5042],
        [ 0.1289, -0.9913,  0.2062]])

Notice that there is no control flow in the graph - we've baked in the value of `flag`.  What if its value changes?

In [8]:
torch._logging.set_logs(graph_code=True, guards=True)

f(torch.randn(3, 3), False)

V1006 16:00:26.395000 14541 torch/_dynamo/output_graph.py:2936] [5/1] [__graph_code] TRACED GRAPH
V1006 16:00:26.395000 14541 torch/_dynamo/output_graph.py:2936] [5/1] [__graph_code]  ===== __compiled_fn_14_6616d78a_0c1f_46ae_a47f_78a0c1eacb2e =====
V1006 16:00:26.395000 14541 torch/_dynamo/output_graph.py:2936] [5/1] [__graph_code]  /home/rob/code/scratch/pytorch/presentation/env/lib/python3.14/site-packages/torch/fx/_lazy_graph_module.py class GraphModule(torch.nn.Module):
V1006 16:00:26.395000 14541 torch/_dynamo/output_graph.py:2936] [5/1] [__graph_code]     def forward(self, L_x_: "f32[3, 3][3, 1]cpu"):
V1006 16:00:26.395000 14541 torch/_dynamo/output_graph.py:2936] [5/1] [__graph_code]         l_x_ = L_x_
V1006 16:00:26.395000 14541 torch/_dynamo/output_graph.py:2936] [5/1] [__graph_code] 
V1006 16:00:26.395000 14541 torch/_dynamo/output_graph.py:2936] [5/1] [__graph_code]         # File: /tmp/ipykernel_14541/2688769548.py:5 in f, code: return x.cos()
V1006 16:00:26.395000 14541 

tensor([[ 0.8680,  0.2544,  0.2786],
        [-0.6422,  0.9960,  0.6869],
        [ 0.6901,  0.9815,  0.7133]])

We need to recompile `f` and make a new FX graph.  Dynamo handles this using guards: a series of conditions which are checked before the compiled function is run.  If any guard fails, the function is recompiled.

Another common source of recompiles is changing tensor shapes

In [9]:
@torch.compile(backend='eager')
def f(x):
    return x.cos()

torch._logging.set_logs(recompiles=True)
f(torch.randn(3, 3))
f(torch.randn(3, 4))

V1006 16:00:27.922000 14541 torch/_dynamo/guards.py:5621] [6/1] [__recompiles] Recompiling function f in /tmp/ipykernel_14541/51478692.py:1
V1006 16:00:27.922000 14541 torch/_dynamo/guards.py:5621] [6/1] [__recompiles]     triggered by the following guard failure(s):
V1006 16:00:27.922000 14541 torch/_dynamo/guards.py:5621] [6/1] [__recompiles]     - 6/0: tensor 'x' size mismatch at index 1. expected 3, actual 4


tensor([[ 0.6529,  0.8454,  0.7268,  0.8387],
        [ 0.7876, -0.8542,  0.0802,  0.9356],
        [ 0.2421,  0.7620, -0.6479,  0.9959]])

Dynamo has a number of tools to work around this.  The simplest is to treat all shapes as dynamic

In [10]:
@torch.compile(backend='eager', dynamic=True)
def f(x):
    return x.cos()

torch._logging.set_logs(recompiles=True)
f(torch.randn(3, 4))
f(torch.randn(3, 5))

tensor([[ 0.9291, -0.0930,  0.7737,  0.2692,  0.9950],
        [-0.4424,  0.2117,  0.6975,  0.7329,  0.9922],
        [ 1.0000, -0.0170,  0.8297,  0.0752,  0.9974]])

# Troubleshooting

With the basics covered, let's talk about troubleshooting more specific problems

### Working around graph breaks

Printing/logging functions are common sources of graph breaks.

In [11]:
@torch.compile(backend='eager', fullgraph=True)
def f(x):
    y = x + 1
    print(y)
    return y + 1

f(torch.ones(3, 3))

Unsupported: Failed to trace builtin operator
  Explanation: Dynamo does not know how to trace builtin operator `print` with argument types ['Tensor'] (has_kwargs False)
  Hint: Avoid calling builtin `print` with argument types ['Tensor']. Consider using an equivalent alternative function/method to `print`.
  Hint: If you are attempting to call a logging function (e.g. `print`), you can try adding it to `torch._dynamo.config.reorderable_logging_functions`.
  Hint: Please report an issue to PyTorch.

  Developer debug context: builtin print [<class 'torch._dynamo.variables.tensor.TensorVariable'>] False

 For more details about this graph break, please visit: https://meta-pytorch.github.io/compile-graph-break-site/gb/gb0059.html

from user code:
   File "/tmp/ipykernel_14541/1303218100.py", line 4, in f
    print(y)
    ~~~~~^^^

Set TORCHDYNAMO_VERBOSE=1 for the internal stack trace (please do this especially if you're reporting a bug to PyTorch). For even more developer context, set T

The error message conveniently mentions `reorderable_logging_functions` as a workaround, which allows Dynamo to move the logging function outside of the graph.  But be careful about mutating the object you're printing!

In [12]:
@torch.compile(backend='eager', fullgraph=True)
def f(x):
    x += 1
    print(x)
    x += 1
    return x

torch._logging.set_logs(graph_code=True)

with torch._dynamo.config.patch(reorderable_logging_functions = {print}):
    f(torch.ones(3, 3))

V1006 16:00:30.295000 14541 torch/_dynamo/output_graph.py:2936] [9/0] [__graph_code] TRACED GRAPH
V1006 16:00:30.295000 14541 torch/_dynamo/output_graph.py:2936] [9/0] [__graph_code]  ===== __compiled_fn_23_123fdeaa_f148_47d5_8914_1241d6146d93 =====
V1006 16:00:30.295000 14541 torch/_dynamo/output_graph.py:2936] [9/0] [__graph_code]  /home/rob/code/scratch/pytorch/presentation/env/lib/python3.14/site-packages/torch/fx/_lazy_graph_module.py class GraphModule(torch.nn.Module):
V1006 16:00:30.295000 14541 torch/_dynamo/output_graph.py:2936] [9/0] [__graph_code]     def forward(self, L_x_: "f32[3, 3][3, 1]cpu"):
V1006 16:00:30.295000 14541 torch/_dynamo/output_graph.py:2936] [9/0] [__graph_code]         l_x_ = L_x_
V1006 16:00:30.295000 14541 torch/_dynamo/output_graph.py:2936] [9/0] [__graph_code] 
V1006 16:00:30.295000 14541 torch/_dynamo/output_graph.py:2936] [9/0] [__graph_code]         # File: /tmp/ipykernel_14541/679276707.py:3 in f, code: x += 1
V1006 16:00:30.295000 14541 torch/_dy

tensor([[3., 3., 3.],
        [3., 3., 3.],
        [3., 3., 3.]])


Alternative options include `torch._dynamo.config.ignore_logging_functions` and `torch._higher_order_ops.print`.

Another common source of graph breaks is using features that Dynamo doesn't understand

In [13]:
def collate(values):
    if hasattr(values, "__len__"):
        return torch.zeros(len(values))
    return torch.zeros(1)


@torch.compile(backend='eager', fullgraph=True)
def f(x):
    doubled = map(lambda v: v * 2, [1.0, 2.0, 3.0])
    return x + collate(doubled).sum()

f(torch.randn(3, 3))

Unsupported: Unsupported hasattr call
  Explanation: Dynamo does not know how to trace the function `MapVariable()`
  Hint: Avoid calling `hasattr(MapVariable, __len__)` in your code.
  Hint: It may be possible to write Dynamo tracing rules for this code. Please report an issue to PyTorch if you encounter this graph break often and it is causing performance issues.

  Developer debug context: call_obj_hasattr MapVariable() __len__

 For more details about this graph break, please visit: https://meta-pytorch.github.io/compile-graph-break-site/gb/gb0150.html

from user code:
   File "/tmp/ipykernel_14541/404762566.py", line 10, in f
    return x + collate(doubled).sum()
               ~~~~~~~^^^^^^^^^
  File "/tmp/ipykernel_14541/404762566.py", line 2, in collate
    if hasattr(values, "__len__"):
       ~~~~~~~^^^^^^^^^^^^^^^^^^^

Set TORCHDYNAMO_VERBOSE=1 for the internal stack trace (please do this especially if you're reporting a bug to PyTorch). For even more developer context, set 

Dynamo doesn't understand how to do attribute lookup on `map` objects.  There's nothing preventing Dynamo from being able to do this, but python is a big language (and a moving target), so it's hard to support everything.  There's ongoing work to improve this by making Dynamo behave more like cpython.  But in the meantime, workarounds are needed

In this case, we can do `list(map(...))` instead.  Lists are more common, and Dynamo has better support for them.

### Recompiles

Recompiles are a common source of problems when compiling larger chunks of code.  Some recompiles are unavoidable (i.e. due to changing control flow).  Others can be worked around

In [14]:
results = {}

@torch.compile(backend='eager', fullgraph=True)
def f(x, name):
    y = x.cos()
    results[name] = y
    return y

for i in range(10):
    f(torch.randn(3,3), f"tensor_{i}")

V1006 16:00:32.429000 14541 torch/_dynamo/output_graph.py:2936] [11/0] [__graph_code] TRACED GRAPH
V1006 16:00:32.429000 14541 torch/_dynamo/output_graph.py:2936] [11/0] [__graph_code]  ===== __compiled_fn_26_84000a37_2f96_4a78_8033_395ee03b594d =====
V1006 16:00:32.429000 14541 torch/_dynamo/output_graph.py:2936] [11/0] [__graph_code]  /home/rob/code/scratch/pytorch/presentation/env/lib/python3.14/site-packages/torch/fx/_lazy_graph_module.py class GraphModule(torch.nn.Module):
V1006 16:00:32.429000 14541 torch/_dynamo/output_graph.py:2936] [11/0] [__graph_code]     def forward(self, L_x_: "f32[3, 3][3, 1]cpu"):
V1006 16:00:32.429000 14541 torch/_dynamo/output_graph.py:2936] [11/0] [__graph_code]         l_x_ = L_x_
V1006 16:00:32.429000 14541 torch/_dynamo/output_graph.py:2936] [11/0] [__graph_code] 
V1006 16:00:32.429000 14541 torch/_dynamo/output_graph.py:2936] [11/0] [__graph_code]         # File: /tmp/ipykernel_14541/89315925.py:5 in f, code: y = x.cos()
V1006 16:00:32.429000 1454

FailOnRecompileLimitHit: Hard failure due to fullgraph=True

Why did we hit the recompile limit?  The error message gives some info, but we can get more by enabling more logging

In [15]:
results = {}

@torch.compile(backend='eager', fullgraph=True)
def f(x, name):
    y = x.cos()
    results[name] = y
    return y

torch._logging.set_logs(guards=True, recompiles=True)

for i in range(3):
    f(torch.randn(3,3), f"tensor_{i}")

V1006 16:00:34.349000 14541 torch/_dynamo/guards.py:5014] [12/0] [__guards] GUARDS:
V1006 16:00:34.350000 14541 torch/_dynamo/guards.py:4693] [12/0] [__guards] 
V1006 16:00:34.350000 14541 torch/_dynamo/guards.py:4693] [12/0] [__guards] TREE_GUARD_MANAGER:
V1006 16:00:34.350000 14541 torch/_dynamo/guards.py:4693] [12/0] [__guards] +- RootGuardManager
V1006 16:00:34.350000 14541 torch/_dynamo/guards.py:4693] [12/0] [__guards] | +- LAMBDA_GUARD: torch._functorch.aot_autograd.utils.top_saved_tensors_hooks ids == None  # _dynamo/output_graph.py:1194 in init_ambient_guards
V1006 16:00:34.350000 14541 torch/_dynamo/guards.py:4693] [12/0] [__guards] | +- GLOBAL_STATE: ___check_global_state() against {"allow_bf16_reduce":0,"allow_fp16_reduce":0,"allow_tf32":false,"autocast_state":{"cached_enabled":true,"dtype":[15,5,5,15,5,5,15,15,5,5],"enabled":[false,false,false,false,false,false,false,false,false,false]},"default_dtype":6,"deterministic_algorithms":false,"deterministic_algorithms_warn_only"

Dynamo treats strings as constants, so it needs to guard on the exact value.  This isn't really fundamental - in this case we could just generate a side effect that updates the dict without needing to guard on the exact string value.  This is an area of future work for Dynamo, but for now this is a necessary recompile.  If you want to avoid it, you can use `is_compiling` to skip the write to `results` in the generated function

In [16]:
results = {}

@torch.compile(backend='eager', fullgraph=True)
def f(x, name):
    y = x.cos()
    if not torch.compiler.is_compiling():
        results[name] = y
    return y

torch._logging.set_logs(recompiles=True)

for i in range(10):
    f(torch.randn(3,3), f"tensor_{i}")
print(f'{results=}')

results={}


### Dynamic shapes

We mentioned setting `dynamic=True` earlier.  This makes all shapes and scalar inputs dynamic by default, but the more common case is that some dimensions are dynamic while other are not.  It's generally preferable to instead mark specific dimensions as dynamic and let the compiler treat the others as static

In [17]:
@torch.compile(backend='eager')
def f(x):
    return x.cos()

x = torch.randn(3, 3)
torch._dynamo.mark_dynamic(x, 1)
torch._logging.set_logs(recompiles=True)
f(x)
f(torch.randn(3, 4))

tensor([[0.5085, 0.5862, 0.7591, 0.9979],
        [0.8837, 0.8720, 0.9798, 0.3526],
        [0.6525, 0.7984, 0.9242, 0.7799]])

Dynamic shapes in particular is a big subject - see the links at the end for more info.

### Escape hatches
Sometimes you want to have code with some sections that are just too hard to compile - they may do things that Dynamo fundamentally doesn't understand, or they may just be too big/complex to be worth fixing.  Escape hatches give you various ways to include these sections.

#### Custom ops

For example, consider a native function that Dynamo doesn't understand

In [18]:
import scipy

@torch.compile(backend='eager', fullgraph=True)
def f(x):
    return scipy.special.ellipk(x.numpy())

f(torch.randn(3,3))

Unsupported: call to a callable object with no traceable __call__
  Explanation: Dynamo could not trace a Python `__call__` method on this object.
  Hint: It may be possible to write Dynamo tracing rules for this code. Please report an issue to PyTorch if you encounter this graph break often and it is causing performance issues.

  Developer debug context: object=<ufunc 'ellipk'>

 For more details about this graph break, please visit: https://meta-pytorch.github.io/compile-graph-break-site/gb/gb4818.html

from user code:
   File "/tmp/ipykernel_14541/1418402443.py", line 5, in f
    return scipy.special.ellipk(x.numpy())
           ~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^

Set TORCHDYNAMO_VERBOSE=1 for the internal stack trace (please do this especially if you're reporting a bug to PyTorch). For even more developer context, set TORCH_LOGS="+dynamo"


I'm using scipy as an example, but in practice this might be any operator written in native code for performance.  The solution is to use a custom op, which allows the compiler to treat the function as a black box.  Importantly, you have to define a fake kernel so the compiler knows what shape the output will be

In [19]:
@torch.library.custom_op(
    "mylib::scipy_ellipk",
    mutates_args=(),
    device_types="cpu",
)
def scipy_ellipk(x: torch.Tensor) -> torch.Tensor:
    return scipy.special.ellipk(x.numpy())


@scipy_ellipk.register_fake
def _(x):
    return torch.empty_like(x)

@torch.compile(backend='eager', fullgraph=True)
def f(x):
    return scipy_ellipk(x.numpy())

torch._logging.set_logs(graph_code=True)
f(torch.randn(3,3))

V1006 16:00:39.928000 14541 torch/_dynamo/output_graph.py:2936] [16/0] [__graph_code] TRACED GRAPH
V1006 16:00:39.928000 14541 torch/_dynamo/output_graph.py:2936] [16/0] [__graph_code]  ===== __compiled_fn_53_d19db2e7_8fd9_44d6_ab87_a6faf1e4480c =====
V1006 16:00:39.928000 14541 torch/_dynamo/output_graph.py:2936] [16/0] [__graph_code]  /home/rob/code/scratch/pytorch/presentation/env/lib/python3.14/site-packages/torch/fx/_lazy_graph_module.py class GraphModule(torch.nn.Module):
V1006 16:00:39.928000 14541 torch/_dynamo/output_graph.py:2936] [16/0] [__graph_code]     def forward(self, L_x_: "f32[3, 3][3, 1]cpu"):
V1006 16:00:39.928000 14541 torch/_dynamo/output_graph.py:2936] [16/0] [__graph_code]         l_x_ = L_x_
V1006 16:00:39.928000 14541 torch/_dynamo/output_graph.py:2936] [16/0] [__graph_code] 
V1006 16:00:39.928000 14541 torch/_dynamo/output_graph.py:2936] [16/0] [__graph_code]         # File: /tmp/ipykernel_14541/3777402561.py:16 in f, code: return scipy_ellipk(x.numpy())
V100

array([[1.5043498, 1.7735826, 1.3925713],
       [1.7264305, 1.4217566, 1.7623689],
       [      nan, 1.5147638, 1.2664446]], dtype=float32)

#### Nonstrict trace

What if you function has some hard to trace code that you don't really care about?  

In [20]:
import json

@torch.compiler.nonstrict_trace
def g(x):
    print(json.dumps({"shape": list(x.shape), "dtype": str(x.dtype)}))
    return x ** 2

@torch.compile(fullgraph=True)
def f(x):
    y = x.cos()
    return g(y) + 1

torch._logging.set_logs(graph_code=True)

print('compilation:')
f(torch.randn(3, 3))
print('2nd call:')
f(torch.randn(3, 3))

V1006 16:00:41.409000 14541 torch/_dynamo/output_graph.py:2936] [17/0] [__graph_code] TRACED GRAPH
V1006 16:00:41.409000 14541 torch/_dynamo/output_graph.py:2936] [17/0] [__graph_code]  ===== __compiled_fn_55_484f6f6b_7250_4253_aa6c_75cbc1c73931 =====
V1006 16:00:41.409000 14541 torch/_dynamo/output_graph.py:2936] [17/0] [__graph_code]  /home/rob/code/scratch/pytorch/presentation/env/lib/python3.14/site-packages/torch/fx/_lazy_graph_module.py class GraphModule(torch.nn.Module):
V1006 16:00:41.409000 14541 torch/_dynamo/output_graph.py:2936] [17/0] [__graph_code]     def forward(self, L_x_: "f32[3, 3][3, 1]cpu"):
V1006 16:00:41.409000 14541 torch/_dynamo/output_graph.py:2936] [17/0] [__graph_code]         l_x_ = L_x_
V1006 16:00:41.409000 14541 torch/_dynamo/output_graph.py:2936] [17/0] [__graph_code] 
V1006 16:00:41.409000 14541 torch/_dynamo/output_graph.py:2936] [17/0] [__graph_code]         # File: /tmp/ipykernel_14541/1314338206.py:10 in f, code: y = x.cos()
V1006 16:00:41.409000 1

compilation:
{"shape": [3, 3], "dtype": "torch.float32"}
{"shape": [3, 3], "dtype": "torch.float32"}
{"shape": [3, 3], "dtype": "torch.float32"}
2nd call:


tensor([[1.5311, 1.3345, 1.5493],
        [1.7758, 1.1447, 1.5561],
        [1.7720, 1.5993, 1.9990]])

Be careful with nonstrict trace!  It disables normal dynamo tracing, which will lose side effects, control flow, etc.  But it is a convenient way to just make a function compile

#### Disabling compile
If you do care about the side effects, you can instead use `torch.compiler.disable`, which will force a graph break and fall back to the normal python compiler.  Your function will execute normally, but you'll lose some optimization opportunities.

In [21]:
@torch.compiler.disable
def g(x):
    print(json.dumps({"shape": list(x.shape), "dtype": str(x.dtype)}))
    return x ** 2

@torch.compile
def f(x):
    y = x.cos()
    return g(y) + 1

torch._logging.set_logs(graph_code=True)

print('compilation:')
f(torch.randn(3, 3))
print('2nd call:')
f(torch.randn(3, 3))

V1006 16:00:45.892000 14541 torch/_dynamo/output_graph.py:2936] [18/0_1] [__graph_code] TRACED GRAPH
V1006 16:00:45.892000 14541 torch/_dynamo/output_graph.py:2936] [18/0_1] [__graph_code]  ===== __compiled_fn_58_ff7049e1_12c2_4a92_82de_3b2f103529c1 =====
V1006 16:00:45.892000 14541 torch/_dynamo/output_graph.py:2936] [18/0_1] [__graph_code]  /home/rob/code/scratch/pytorch/presentation/env/lib/python3.14/site-packages/torch/fx/_lazy_graph_module.py class GraphModule(torch.nn.Module):
V1006 16:00:45.892000 14541 torch/_dynamo/output_graph.py:2936] [18/0_1] [__graph_code]     def forward(self, L_x_: "f32[3, 3][3, 1]cpu"):
V1006 16:00:45.892000 14541 torch/_dynamo/output_graph.py:2936] [18/0_1] [__graph_code]         l_x_ = L_x_
V1006 16:00:45.892000 14541 torch/_dynamo/output_graph.py:2936] [18/0_1] [__graph_code] 
V1006 16:00:45.892000 14541 torch/_dynamo/output_graph.py:2936] [18/0_1] [__graph_code]         # File: /tmp/ipykernel_14541/4004133733.py:8 in f, code: y = x.cos()
V1006 16:0

compilation:


V1006 16:00:46.390000 14541 torch/_dynamo/output_graph.py:2936] [19/0] [__graph_code] TRACED GRAPH
V1006 16:00:46.390000 14541 torch/_dynamo/output_graph.py:2936] [19/0] [__graph_code]  ===== __compiled_fn_61_4daad320_cd0f_469a_827b_f567f864ee2f =====
V1006 16:00:46.390000 14541 torch/_dynamo/output_graph.py:2936] [19/0] [__graph_code]  /home/rob/code/scratch/pytorch/presentation/env/lib/python3.14/site-packages/torch/fx/_lazy_graph_module.py class GraphModule(torch.nn.Module):
V1006 16:00:46.390000 14541 torch/_dynamo/output_graph.py:2936] [19/0] [__graph_code]     def forward(self, L_stack0_: "f32[3, 3][3, 1]cpu"):
V1006 16:00:46.390000 14541 torch/_dynamo/output_graph.py:2936] [19/0] [__graph_code]         l_stack0_ = L_stack0_
V1006 16:00:46.390000 14541 torch/_dynamo/output_graph.py:2936] [19/0] [__graph_code] 
V1006 16:00:46.390000 14541 torch/_dynamo/output_graph.py:2936] [19/0] [__graph_code]         # File: /tmp/ipykernel_14541/4004133733.py:9 in torch_dynamo_resume_in_f_at_9,

{"shape": [3, 3], "dtype": "torch.float32"}
2nd call:
{"shape": [3, 3], "dtype": "torch.float32"}


tensor([[1.9909, 1.6812, 1.8287],
        [1.9836, 1.0052, 1.8495],
        [1.0223, 1.4797, 1.6627]])

### Higher order ops

One of the fundamental constructs that Dynamo doesn't support is data dependent control flow.  Remember - FX graphs don't support control flow.  Normally we can get around that by "baking in" which branch is taken, but not if the condition is dependent on e.g. tensor data.

In [22]:
@torch.compile(backend='eager', fullgraph=True)
def f(x):
    if x.sum() > 0:
        return x.cos()
    return x.sin()

f(torch.randn(3, 3))

Unsupported: Data-dependent branching
  Explanation: Detected data-dependent branching (e.g. `if my_tensor.sum() > 0:`). Dynamo does not support tracing dynamic control flow.

      The branch condition involves a tensor computed as follows:
        # File "/tmp/ipykernel_14541/1157727851.py", line 3, in f, code: if x.sum() > 0:
        gt = gt(sum_1, 0)
    
  Hint: For the common pattern `if tensor_cond: x = transform(x)` (e.g. clamping inf/nan values), consider making the code branchless by always applying the transform. Operations like torch.clamp, torch.nan_to_num, and torch.where are typically no-ops on well-behaved inputs and compile without graph breaks.
  Hint: This graph break is fundamental - it is unlikely that Dynamo will ever be able to trace through your code. Consider finding a workaround.
  Hint: Use `torch.cond` to express dynamic control flow.

  Developer debug context: attempted to jump with TensorVariable()

 For more details about this graph break, please visit: 

Higher order operators (aka HOPs) allow you to express this in a way that torch.compile can understand

In [23]:
@torch.compile(backend='eager', fullgraph=True)
def f(x):
    return torch.cond(x.sum() > 0, lambda x: x.cos(), lambda x: x.sin(), (x,))

torch._logging.set_logs(graph_code=True)

f(torch.randn(3, 3))

V1006 16:00:49.502000 14541 torch/_dynamo/output_graph.py:2936] [21/0] [__graph_code] TRACED GRAPH
V1006 16:00:49.502000 14541 torch/_dynamo/output_graph.py:2936] [21/0] [__graph_code]  ===== __compiled_fn_64_527efa3e_b173_45d0_a109_6ddcdaea9eda =====
V1006 16:00:49.502000 14541 torch/_dynamo/output_graph.py:2936] [21/0] [__graph_code]  /home/rob/code/scratch/pytorch/presentation/env/lib/python3.14/site-packages/torch/fx/_lazy_graph_module.py class GraphModule(torch.nn.Module):
V1006 16:00:49.502000 14541 torch/_dynamo/output_graph.py:2936] [21/0] [__graph_code]     def forward(self, L_x_: "f32[3, 3][3, 1]cpu"):
V1006 16:00:49.502000 14541 torch/_dynamo/output_graph.py:2936] [21/0] [__graph_code]         l_x_ = L_x_
V1006 16:00:49.502000 14541 torch/_dynamo/output_graph.py:2936] [21/0] [__graph_code] 
V1006 16:00:49.502000 14541 torch/_dynamo/output_graph.py:2936] [21/0] [__graph_code]         # File: /tmp/ipykernel_14541/3628540505.py:3 in f, code: return torch.cond(x.sum() > 0, lambd

tensor([[ 0.8393, -0.4377, -0.4826],
        [-0.7029, -0.5761,  0.9217],
        [ 0.7294, -0.2364, -0.6417]])

More dynamo resources:

- [Overview](https://docs.pytorch.org/docs/2.14/user_guide/torch_compiler/torch.compiler_dynamo_overview.html)
- [Deep dive](https://docs.pytorch.org/docs/2.14/user_guide/torch_compiler/torch.compiler_dynamo_deepdive.html)
- [Troubleshooting](https://docs.pytorch.org/docs/2.14/user_guide/torch_compiler/torch.compiler_troubleshooting.html)
- [Fine-grained tracing APIs](https://docs.pytorch.org/docs/2.14/user_guide/torch_compiler/torch.compiler_fine_grain_apis.html)
- [Dynamic shapes](https://docs.pytorch.org/docs/2.14/user_guide/torch_compiler/torch.compiler_dynamic_shapes.html)